# Notebook 02: Preprocessing, Feature Engineering & Event Integration

Load the consolidated, provenance-preserving datasets and prepare their common event schema.

In [1]:
from pathlib import Path
import sys
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name.lower() == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.preprocessing import prepare_event_dataframe
from src.features import detect_behaviours

synthetic_dir = PROJECT_ROOT / "data" / "synthetic"
processed_dir = PROJECT_ROOT / "data" / "processed"
synthetic_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)
communication = pd.read_csv(synthetic_dir / "communication_events.csv", dtype={"sender_number": str})
transactions = pd.read_csv(synthetic_dir / "transactions.csv")

assert communication["source_dataset"].nunique() == 3
assert transactions["source_dataset"].nunique() == 2
print("Communication shape:", communication.shape)
print("Communication source rows:", communication["source_dataset"].value_counts().to_dict())
print("Transaction shape:", transactions.shape)
print("Transaction source rows:", transactions["source_dataset"].value_counts().to_dict())

Communication shape: (1300, 15)
Communication source rows: {'data\\synthetic\\communication_events.csv': 500, 'notebooks\\data\\synthetic\\communication_events.csv': 500, 'notebooks\\data\\synthetic\\communication_synthetic.csv': 300}
Transaction shape: (2000, 20)
Transaction source rows: {'data\\synthetic\\transactions.csv': 1000, 'notebooks\\data\\synthetic\\transaction_behaviour_synthetic.csv': 1000}


In [2]:
communication = prepare_event_dataframe(communication)
display(communication[["event_id", "event_type", "clean_message", "is_fraudulent"]].head(10))

,event_id,event_type,clean_message,is_fraudulent
0,L0119,SMS,this is a confirmation that the payment was co...,0
1,L0100,call,your otp was used to approve your recent accou...,0
2,L0225,SMS,your cash out request was successful and a rec...,0
3,F0025,SMS,send the verification code within five minutes...,1
4,L0042,SMS,the charge on your account matches the merchan...,0
5,L0130,SMS,your transaction of ghc100 was successful,0
6,F0208,call,call our helpline now and provide the code app...,1
7,L0153,SMS,the recipient confirmed receipt of the funds,0
8,L0160,call,please contact official support only through t...,0
9,F0134,SMS,we detected suspicious activity do not contact...,1


In [3]:
communication["detected_behaviours"] = communication["message"].map(detect_behaviours)
display(communication[["event_id", "detected_behaviours", "is_fraudulent"]].head(15))

,event_id,detected_behaviours,is_fraudulent
0,L0119,[],0
1,L0100,[],0
2,L0225,[],0
3,F0025,[],1
4,L0042,[],0
5,L0130,[],0
6,F0208,[],1
7,L0153,[],0
8,L0160,[],0
9,F0134,[],1


## Behavioural features

Preserve the transaction context separately from text-derived fraud indicators.

In [ ]:
behavioural_columns = [
    "amount", "is_new_recipient", "recent_suspicious_sms",
    "recent_suspicious_call", "otp_request", "new_device",
]

transactions.to_csv(processed_dir / "transactions_features.csv", index=False)
display(transactions.groupby("is_fraudulent")[behavioural_columns].mean().round(3))
print("Saved:", processed_dir / "transactions_features.csv")